This is a conversational RAG application using LangChain and a Huggingface (open source) LLM model. It can perform the conversation based on LLM model, and other additional project data which we input into the RAG architecture. Furthermore it can maintain a cha t history to memorize previous chats and repsonse accordingly 

In [ ]:
#Install required packages

#To use the deepe learning (Neural network) whiel using huggingface
!pip install torch -q
!pip install numpy -q
#to use pretrained AI models
!pip install transformers -q
#huggingface
!pip install -U langchain-huggingface
#langchain framework, chains gents etc
!pip install langchain -q
#To use chroma vectordatabase
!pip install langchain-chroma -q
#To use other thrid party product integration 
!pip install langchain_community -q
#to convert sentecse or docs to embedding
!pip install sentence_transformers -q


Initialize the Huggingface LLM. We are going to use the google flan model, alos can use lama, falcon etc

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="conversational",
    huggingfacehub_api_token="hf_API_token",#add the API key here 
    temperature=0.1,
    max_new_tokens=100
)

In [ ]:
llm.invoke("Explain AI in one sentence.")

In [ ]:
#initialize the embedding model
#we use a sentence_transformers embedding model
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
#Initialize the Output parser
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()

In [ ]:
#load pdf docuemnt
from langchain_community.document_loaders import PyPDFLoader
#load the PDF Document
loader=PyPDFLoader("/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/IPC_2025_Paper_final_version.pdf")
docs=loader.load()

In [ ]:
#checking the loaded document
len(docs) #gives no of pages

In [ ]:
#first page data, contnent and meta data
docs[0]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#intialize the text splitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
#split the documents into chunks
splits=text_splitter.split_documents(docs)

In [ ]:
len(splits)

In [ ]:
splits[0]

In [ ]:
#Create the vector store and the retriever
from langchain_chroma import Chroma

#Create a vector store from the document chunks
vectorStore=Chroma.from_documents(documents=splits,embedding=embedding_model)

#Create the retirever
retriever=vectorStore.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are an intelligent chatbot. Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
prompt # gieve the prompt detaisl, here the context is coming from retirever,and the input is the question from the user

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# RAG chain using  retirever+QA chain  where QA chain=llm+prompt, here runnablrPassthrough is used to pass the orginal question from the user to the chain
rag_chain = (
    {"context": retriever, "input": RunnablePassthrough()} # retirever + input
    | prompt #make ready the human input,retirever and system prompt to feed to llm
    | llm #actual model
    | output_parser #llm returns and message, parser extract just the string instrad of that object
)

In [ ]:
print(type(llm))

In [ ]:
#invoke RAG chain with example questions
response=rag_chain.invoke("what are polarization aware modular IR clusters")
print(response)

In [ ]:
response2=rag_chain.invoke("what is RAG architecture")
print(response2) #Since given pdf article doesnt include those detials, it doesnt know, it, and says it doesnt know.

In [ ]:
response3=rag_chain.invoke("how we reduce the interfernence in them") #seems like doesnt maintin abeteer chat history yet to answer
print(response3)

Adding chat hisotry

In [ ]:
#Correct imports for langchain v1.x
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

contextualize_system_prompt = (
    "Given the chat history and the latest user question, "
    "reformulate the question to be standalone if needed. "
    "Otherwise return it as is. Do NOT answer the question."
)

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

#history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

#QA prompt
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an intelligent chatbot. Use the context to answer. "
               "If you don't know, say you don't know.\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

#Full RAG chain
document_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

In [ ]:
#Manage cht session history
#base on the session id we prepare a discionary to store the chat history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#initialize the store dict for session histories
store={}# this dictonary can store ina database or jason fiel later

#function to get the session history for a given session ID
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

#create the conversational RAG chain with session history
conversational_rag_chain=RunnableWithMessageHistory(rag_chain,get_session_history,input_messages_key="input",history_messages_key="chat_history",output_messages_key="answer",)


In [ ]:
#invoke the conversational_rag_chain with example questions
response=conversational_rag_chain.invoke({"input":"what are IR radiative clusters"},config={"configurable":{"session_id":"101"}},)
response["answer"]

In [ ]:
response=conversational_rag_chain.invoke({"input":"how polarization join here"},config={"configurable":{"session_id":"101"}},)
response["answer"] #so now answer has given how polarization can manage for previously mentioned cluster IR arrays

In [ ]:
response=conversational_rag_chain.invoke({"input":"how polarization join here"},config={"configurable":{"session_id":"103"}},)
response["answer"] #so now answer has given how polarization can manage for previously mentioned cluster IR arrays